# Plot carbon stocks blue trendy literature (figure m06)

In [ ]:
# Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Rectangle
from matplotlib.legend_handler import HandlerTuple

In [ ]:
# Directories
dir_zen = '../paper_ecosystem_carbon/output/zenodo/'
dir08 = '../paper_ecosystem_carbon/output/08_eval/'

---

### Data preparation

In [ ]:
# Prepared dataframe with blue, dgvm and literature data
df_in = pd.read_csv(f'{dir_zen}ecosystem_carbon_data_figure_m06.csv')

In [ ]:
# Dictionary with trendy models and original model names
dict_trendy_names = {
    'cablepop': 'CABLEPOP',
    'classic': 'CLASSIC',
    'clm': 'CLM5.0',
    'dlem': 'DLEM',
    'ibis': 'IBIS',
    'isam': 'ISAM',
    'jsbach': 'JSBACH',
    'jules': 'JULES',
    'lpj-guess': 'LPJ-GUESS',
    'lpjml': 'LPJmL',
    'lpxbern': 'LPX-Bern',
    'ocn': 'OCN',
    'orchidee': 'ORCHIDEE',
    'sdgvm': 'SDGVM',
    'yibs': 'YIBS',
    'trendymean': 'TRENDY mean'}

if set(
    i.split('_')[0] for i in 
    df_in[df_in['source_type'] == 'trendyv12']['source']
) - set(dict_trendy_names):
    raise ValueError(
        'Mismatch between trendy models in df_cpools and dict_trendy_names'
    )

# Dictionary with literature names and labels
dict_lit = {
    'spawn': ('Spawn et al. 2020', 7),
    'erb': ('Erb et al. 2018 (mean)', 5),
    'walker': ('Walker et al. 2022 (mean)', 6),
    'sanderman': ('Sanderman et al. 2017', 2),
    'batjes': ('WISE30sec (Batjes 2016)', 1),
    'tifafi-soilgrids': ('SoilGrids 2017', 4),
    'tifafi-hwsd': ('HWSD 2012', 3),
    }

---

### Plot functions

In [ ]:
def get_v_cpools(m, year):
    """Returns model values for cveg and csoil"""
    df_sel = df_in[(df_in['source'] == f'{m}_{year}')]
    v_cveg = df_sel[(df_sel['carbon_type'] == 'cveg')].value_pgc.item()
    v_csoil = df_sel[(df_sel['carbon_type'] == 'csoil')].value_pgc.item()
    
    return v_cveg, v_csoil 

In [ ]:
def add_point_blue(m, year, marker, s, zorder=5):
    """Adds blue data to plot"""  
    v_cveg, v_csoil = get_v_cpools(m, year)
    
    plt.scatter(v_csoil, v_cveg, c=dict_color[m], s=s, marker=marker,
                edgecolors='#000000', linewidth=1, zorder=zorder)

In [ ]:
def add_arrow_blue(m, zorder=5):
    
    """Draws an arrow between start and end points"""
    
    v_cveg1800, v_csoil1800 = get_v_cpools(m, 1800)
    v_cveg2022, v_csoil2022 = get_v_cpools(m, 2022)

    delta_csoil = v_csoil2022 - v_csoil1800
    delta_cveg = v_cveg2022 - v_cveg1800
    length = np.sqrt(delta_csoil**2 + delta_cveg**2)

    # Normalize the direction vector
    dx = delta_csoil / length
    dy = delta_cveg / length
    offset = length * 0.05
    start_x = v_csoil1800 + dx * offset
    start_y = v_cveg1800 + dy * offset
    end_x = v_csoil2022 - dx * offset
    end_y = v_cveg2022 - dy * offset

    # Define arrow properties
    arrow_props = dict(color=dict_color[m], linewidth=2,
                       arrowstyle='simple,head_width=1.2,head_length=1.2',
                       connectionstyle='arc3,rad=0')
    # Draw arrow
    plt.annotate("", xy=(end_x, end_y), xytext=(start_x, start_y),
                 arrowprops=arrow_props, zorder=zorder)

In [ ]:
def add_point_trendy(m, zorder=5):

    """Adds trendy data to plot"""
    
    # Get data
    v_cveg, v_csoil = get_v_cpools(m, 2022)

    # Add point
    if m == 'trendymean':
        plt.scatter(v_csoil, v_cveg, c=color_trendy, s=120, marker='p',
                        edgecolors=color_trendy2, linewidth=1, zorder=zorder)
    else:
        plt.scatter(v_csoil, v_cveg, c=color_trendy, s=120, marker='o',
                    edgecolors=color_trendy2, linewidth=1, zorder=zorder)

    
    def add_label_trendy(m, x_offset, y_offset):

        # Get data
        v_cveg, v_csoil = get_v_cpools(m, 2022)
        
        # Add label
        plt.text(v_csoil + x_offset, v_cveg + y_offset, dict_trendy_names[m], 
                 color=dict_color['trendy'],
                 fontsize=8, #fontweight='bold', 
                 ha='center', va='center', 
                 zorder=zorder)

    l_labels = {
        'dlem': (0, 15),
#        'trendymean': (0, 15),
        'jules': (-125, 0),
        'cablepop': (-30, 15),
        'ibis': (0, 15),
        'sdgvm': (0, 15),
        'jsbach': (0, -15),
        'orchidee': (0, -15),
        'classic': (-10, 15),
        'clm': (150, 0),
        'isam': (120, 0),
        'yibs': (110, 0),
        'lpj-guess': (-180, -7),
        'lpjml': (-130, 0),
        'ocn': (-40, -17),
        'lpxbern': (175, -7),
    }
    
    for name, (dx, dy) in l_labels.items():
        add_label_trendy(name, dx, dy)

In [ ]:
def add_lit_line(m):

    """Adds literature data as lines to plot"""

    df_sel = df_in[(df_in['source'] == f'{m}')]

    line_kwargs = dict(color='#000000', ls='--', lw=1)
    text_kwargs = dict(s=dict_lit[m][1], fontsize=10, fontweight='bold', 
                       ha='center', va='center')
    
    if m in ['spawn', 'erb', 'walker'] :
        v = df_sel[(df_sel['carbon_type'] == 'cveg')].value_pgc.item()
        plt.axhline(v, **line_kwargs)
        plt.text(x=3740, y= v + 8 if m =='erb' else v - 10, **text_kwargs)

    if m in ['sanderman', 'batjes', 'tifafi-soilgrids', 'tifafi-hwsd']:
        v = df_sel[(df_sel['carbon_type'] == 'csoil')].value_pgc.item()
        plt.axvline(v, **line_kwargs)
        plt.text(x=v-35, y=738, **text_kwargs)

In [ ]:
def add_reference_point():
    """Adds reference point to plot"""
    df_sel = df_in[(df_in['source'] == 'reference')]
    v_cveg = df_sel[(df_sel['carbon_type'] == 'cveg')].value_pgc.item()
    v_csoil = df_sel[(df_sel['carbon_type'] == 'csoil')].value_pgc.item()
    
    plt.scatter(x=v_csoil, y=v_cveg, s=70, marker='d', color='#000000')

In [ ]:
def add_legend():

    """Adds legend to plot"""
    
    dict_legend_points = {
        '1800': {**dict(marker='s', markersize=7), 
                 'markerfacecolor': '#ffffff', 'markeredgecolor': '#000000',},
        '2022': {**dict(marker='o', markersize=10),
            'markerfacecolor': '#ffffff', 'markeredgecolor': '#000000',},
        'BLUE, cHansis': {**dict(marker='o', markersize=10),
            'markerfacecolor': color_chansis, 'markeredgecolor': '#000000',},
        'BLUE, cSpatial': {**dict(marker='o', markersize=10),
            'markerfacecolor': color_cspatial, 'markeredgecolor': '#000000',},
        'BLUE, cDGVM': {**dict(marker='o', markersize=10),
            'markerfacecolor': color_cdgvm, 'markeredgecolor': '#000000',},
        'TRENDY models': {**dict(marker='o', markersize=10),
            'markerfacecolor': color_trendy, 'markeredgecolor': color_trendy2,},
        'TRENDY ensemble mean': {**dict(marker='p', markersize=10),
            'markerfacecolor': color_trendy, 'markeredgecolor': color_trendy2,},
        'Reference estimate': {**dict(marker='d', markersize=7),
            'markerfacecolor': '#000000', 'markeredgecolor': '#000000',},
    }

    lit_keys = ['batjes', 'sanderman', 'tifafi-hwsd', 
                'tifafi-soilgrids', 'erb', 'walker', 'spawn']
    
    legend_elements = [
        Line2D([0], [0], label=label, **i, linestyle='None', color='#000000') 
        for label, i in dict_legend_points.items()
    ]
    
    legend_elements.extend(
        Line2D([0], [0], 
               marker=f"${dict_lit[i][1]}$", linestyle='None', 
               color='#000000', markersize=7, label=dict_lit[i][0]) 
        for i in lit_keys)

    plt.legend(
    handles=legend_elements,
    loc='upper center',
    bbox_to_anchor=(0.5, -0.13),
    ncol=3,
    facecolor='#ffffff',
    edgecolor='none',
    framealpha=0,
    )

---

### Plot

In [ ]:
# Define colors
color_chansis = '#3274A1'
color_cspatial = '#E1812C'
color_cdgvm = '#3A923A'
color_trendy = '#bdbdbd'
color_trendy2 = '#808080'

dict_color = dict(blue_chansis = color_chansis, 
                  blue_cspatial = color_cspatial, 
                  blue_cdgvm = color_cdgvm, 
                  trendy = color_trendy)

In [ ]:
plt.figure(figsize=(9, 6.5), dpi=600)

# Add blue simulation data
for m in ['blue_chansis', 'blue_cspatial', 'blue_cdgvm']:
    add_point_blue(m, 1800, 's', 50)
    add_point_blue(m, 2022, 'o', 120)
    add_arrow_blue(m, zorder=4)

# Add trendy data
for m in dict_trendy_names.keys():
    add_point_trendy(m)

# Add literature data
for m in dict_lit.keys():
    add_lit_line(m)

# Add reference point
add_reference_point()

# Add legend
add_legend()

# Layout
plt.xlim(None, 3780)
plt.ylim(None, 750)
plt.xlabel("Soil organic carbon (PgC)")
plt.ylabel("Vegetation carbon (PgC)")
plt.tight_layout()

# Export
fstr_out = 'fig_m06_cstocks'
plt.savefig(f'{dir08}pdf/{fstr_out}.pdf', dpi=300, 
            bbox_inches='tight', pad_inches=0.1); 
plt.savefig(f'{dir08}png/{fstr_out}.png', dpi=300, 
            bbox_inches='tight', pad_inches=0.1);